# Assignment 1: Information Retrieval

## Overview
In this assignment you will implement a BM25 retrieval system from scratch.
Given a text query, your system must find the most relevant documents from the **Cranfield Collection** — 
a foundational corpus for evaluating information retrieval systems containing:
- **1,400** documents
- **225** queries
- **1,837** relevance judgments (qrels)

## Rules
- You may only use **built-in Python libraries** plus `numpy` and `pandas`
- Full-packaged retrieval solutions (e.g. `rank_bm25`) are **not permitted**
- Do **not** modify function signatures or the skeleton class
- Do **not** change cells marked **do not edit**

## Deadline
**March 19, 23:59 CET**

## How to Use This Notebook
1. Run imports and BM25 Skeleton class first — always
2. Implement each task in the cells marked **TODO: your code here** and answer questions marked **TODO**
3. If you restart the kernel, re-run all cells from top to bottom

## Load the data

In [2]:
import ir_datasets
dataset = ir_datasets.load("cranfield")

### Inspect the queries

In [3]:
for query in dataset.queries_iter():
    print(query) # namedtuple<query_id, text>
    break

[INFO] [starting] http://ir.dcs.gla.ac.uk/resources/test_collections/cran/cran.tar.gz
[INFO] [finished] http://ir.dcs.gla.ac.uk/resources/test_collections/cran/cran.tar.gz: [00:00] [507kB] [197MB/s]
                                                                                              

GenericQuery(query_id='1', text='what similarity laws must be obeyed when constructing aeroelastic models\nof heated high speed aircraft .')


### Inspect the documents

In [4]:
for doc in dataset.docs_iter():
    print(doc) # namedtuple<doc_id, title, text, author, bib>
    break

CranfieldDoc(doc_id='1', title='experimental investigation of the aerodynamics of a\nwing in a slipstream .', text='experimental investigation of the aerodynamics of a\nwing in a slipstream .\n  an experimental study of a wing in a propeller slipstream was\nmade in order to determine the spanwise distribution of the lift\nincrease due to slipstream at different angles of attack of the wing\nand at different free stream to slipstream velocity ratios .  the\nresults were intended in part as an evaluation basis for different\ntheoretical treatments of this problem .\n  the comparative span loading curves, together with\nsupporting evidence, showed that a substantial part of the lift increment\nproduced by the slipstream was due to a /destalling/ or\nboundary-layer-control effect .  the integrated remaining lift\nincrement, after subtracting this destalling lift, was found to agree\nwell with a potential flow theory .\n  an empirical evaluation of the destalling effects was made for\nthe s

### Inspect the query-document relevance judgments

In [5]:
for qrel in dataset.qrels_iter():
    print(qrel) # namedtuple<query_id, doc_id, relevance, iteration>
    break

TrecQrel(query_id='1', doc_id='184', relevance=2, iteration='0')


## Setup

The cells below set up the imports and define the BM25 skeleton class. 
You do not need to modify anything here — simply run both cells before starting Task 1.

In [6]:
from __future__ import annotations

import math
import re
from collections import Counter, defaultdict
from dataclasses import dataclass
from typing import Dict, Iterable, List, Tuple, Set, Optional

import numpy as np

# NLTK for tokenization
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
nltk.download("stopwords", quiet=True)

True

**Note:** The `BM25` class below contains all the methods you will implement throughout this assignment.
Each task will ask you to implement one or more of these methods and attach them to the class.

In [7]:
# BM25 Skeleton (run this first, do not edit)

class BM25:
    """
    BM25 class:
      - tokenizer: tokenize(text) -> List[str]
      - retriever: retrieve(query_text, k) -> ranked list
      - evaluator: evaluate(metric, k) and grid_search over k1, b

    Brute-force scoring over all documents (no inverted index).
    """
    def __init__(self, k1: float = 1.2, b: float = 0.75, remove_stopwords: bool = True):
        
        # --- BM25 Hyperparameters ---
        self.k1: float = k1                    # term frequency saturation
        self.b: float = b                      # document length normalization
        self.remove_stopwords: bool = remove_stopwords
        
        # Initializations for tokenization
        self._stemmer = PorterStemmer()      # work stemming
        self._stopwords = set(stopwords.words("english")) if self.remove_stopwords else set() # stopwords set
        
        # --- Corpus Statistics (populated by build()) ---
        self.N: int = 0                        # total number of documents
        self.avgdl: float = 0.0               # average document length
        self.df: Dict[str, int] = {}          # term -> number of docs containing term
        self.doc_len: Dict[str, int] = {}     # doc_id -> document length
        self.doc_tf: Dict[str, Counter] = {}  # doc_id -> term frequency Counter
        self.docs_store: Dict[str, Dict[str, str]] = {}  # doc_id -> {"title": ..., "text": ...}

    def tokenize(self, text: str) -> List[str]: raise NotImplementedError
    def build(self, docs: Iterable) -> None: raise NotImplementedError
    def idf(self, term: str) -> float: raise NotImplementedError
    def score(self, query_tokens: List[str], doc_id: str) -> float: raise NotImplementedError
    def retrieve(self, query_text: str, k: int = 10) -> List[Tuple[str, float]]: raise NotImplementedError
    def evaluate(self, queries_iter: Iterable, qrels_iter: Iterable, k: int = 10, metrics: Optional[List[str]] = None) -> Dict[str, float]: raise NotImplementedError
    def grid_search(self, queries_iter_factory, qrels_iter_factory, k: int = 10) -> List[Dict]: raise NotImplementedError



## Task 1: Tokenizer 

Implement a tokenizer that converts a text into a list of keywords: tokenize(text) -> List[str].

In [30]:
def tokenize(self, text: str) -> List[str]:
    """
    Convert raw text into a list of processed tokens.

    Requirements:
      1. Lowercase everything         e.g.  "UZH"                       -> ["uzh"]
      2. Extract alphanumeric words   e.g.  "hello, world 2026!"        -> ["hello", "world", "2026"]
      3. Remove stopwords             e.g.  "the document has a title"  -> ["document", "title"]
      4. Apply Porter stemming        e.g.  "working worked works"      -> ["work", "work", "work"]
      5. Drop empty tokens            e.g.  " retrieval "               -> ["retrieval"]

    Args:
        text: raw input string
    Returns:
        List of processed token strings

    Hints:
        - Perform text normalization using Python string methods.
        - Use the `re` module to extract alphanumeric tokens.
        - For stemming and stopword removal, use the initialized stemmer and stopword list from the BM25 class constructor.

    """
    tokenized_text: list[str]
    text = text.lower()
    text = re.sub(r'[\W_]', ' ', text)
    tokenized_text = [word for word in text.split() if word not in self._stopwords]
    return [self._stemmer.stem(word) for word in tokenized_text]
    
# attach to BM25 class
BM25.tokenize = tokenize

In [32]:
# ===============================================
# NOTE: Personal Testing Only — NOT Part of Assignment
# ===============================================
#
# This cell contains some basic example test cases for the `tokenize()` method.
# It is meant to help you check your own implementation.
# 
# It does NOT count toward your grade and is safe to run multiple times.
# You can modify it to test additional examples if you want.


bm25_test = BM25(remove_stopwords=True)

# Test 1: Lowercasing + stemming
assert bm25_test.tokenize("Working worked works") == ["work", "work", "work"]

# Test 2: Alphanumeric extraction + punctuation removal
assert bm25_test.tokenize("Hello, world 2026!") == ["hello", "world", "2026"]

# Test 3: Stopword removal + stemming
assert bm25_test.tokenize("The document has a title") == ["document", "titl"]

# Test 4: Mixed case + symbols + stemming
assert bm25_test.tokenize("Machine-Learning @ UZH :)") == ["machin", "learn", "uzh"]

print("All tokenizer tests passed")

All tokenizer tests passed


## Task 2: BM25 Retriever

Implement the document retriever using BM25 that finds the best top-k documents for a given query.

### Task 2a: Build Index

Implement `build()` to precompute the corpus statistics needed for BM25 scoring.

Once complete, the following attributes will be populated and used by `score()` and `retrieve()` in the next steps.

In [ ]:

def build(self, docs_iter: Iterable) -> None:
    """
    Iterate over all documents and precompute corpus statistics needed for BM25.

    Args:
        docs_iter: iterable of Cranfield documents, each a namedtuple with fields:
                   (doc_id, title, text, author, bib)
    
    Requirements: 
        After calling build(), the following must be populated:
            self.N        : int                      — total number of documents
            self.avgdl    : float                    — average document length
            self.df       : Dict[str, int]           — term -> number of docs containing term
            self.doc_len  : Dict[str, int]           — doc_id -> document length
            self.doc_tf   : Dict[str, Counter]       — doc_id -> term frequencies per document
            self.docs_store: Dict[str, Dict[str, str]] — doc_id:str -> {"title": ..., "text": ...} (raw title & text per document)

    Note:
        Per-document steps (repeat for each document):
            1. Extract title and text safely
            2. Concatenate title and text, then tokenize using self.tokenize()
            3. Compute term frequencies  → store in self.doc_tf
            4. Compute document length → store in self.doc_len
            5. Store raw title and text → store in self.docs_store

        Corpus-level steps (after all documents):
            6. Count how many documents each term appears in → store in self.df
            (each term is counted once per document, not once per occurrence)
            7. Compute self.avgdl and self.N

    """

     #TODO: your code here
    pass

# attach to BM25 class
BM25.build = build

### Task 2b: BM25 Scoring

Now that the corpus statistics can be precomputed, implement the two core scoring functions of BM25.

- `idf(term)` computes how rare a term is across the corpus
- `score(query_tokens, doc_id)` combines IDF with term frequency to compute a BM25 score for a query-document pair


In [ ]:
def idf(self, term: str) -> float:
    """
    Compute Inverse Document Frequency with standard smoothing.

    Args:
        term: a single token str

    Returns:
        float — IDF score for the term

    Formula:
        log( (N - df + 0.5) / (df + 0.5) + 1 )

    """

    # TODO: your code here 
    # implement IDF formula
    pass

# attach to BM25 class
BM25.idf = idf


def score(self, query_tokens: List[str], doc_id: str) -> float:
    """
    Compute BM25 score for a query against a single document.

    Args:
        query_tokens: list of tokenized query terms
        doc_id: document to score against

    Returns:
        float — BM25 score for this query-document pair

    Formula (sum over query terms):
        score(q, d) = sum over t in q [ IDF(t) * (f * (k1 + 1)) / (f + k1 * (1 - b + b * dl/avg_dl)) ]

    where:
        q    = query
        d    = document
        t    = a query term
        f    = term frequency of t in document
        dl   = length of the document
        k1   = self.k1
        b    = self.b

    Hints:
        - skip terms with frequency 0
        - return 0.0 for empty documents
        - sum over query terms (you can keep duplicates, but set() is common and stable for students)
    """

    # TODO: your code here
    pass


# attach to BM25 class
BM25.score = score

### Task 2c: Retrieve

Implement `retrieve()` to return the top-k most relevant documents for a given query, ranked by BM25 score.

In [ ]:
def retrieve(self, query_text: str, k: int = 10) -> List[Tuple[str, float]]:
    """
    Return the top-k documents for a given query, ranked by BM25 score.
    Brute-force over all documents.
    
    Args:
        query_text: raw query string
        k:          number of documents to return

    Returns:
        List of (doc_id, score) tuples sorted by score descending

    Hints:
        - tokenize the query first using self.tokenize()
        - iterate over all documents in self.doc_tf.keys()
        - use self.score() to compute the BM25 score for each document
        - only keep documents with a non-zero score
    """

    # TODO: your code here

    pass

# attach to BM25 class
BM25.retrieve = retrieve

### Check Task 2: Implementing a BM25 retriever

Run the cell below to check your implementation. You should see:
- Total number of documents and average document length after building the index
- A ranked list of 10 documents for a sample query with their BM25 scores and titles


In [ ]:
# Build index
bm25 = BM25()
print("Building index...")
bm25.build(dataset.docs_iter())
print(f"Done. N={bm25.N} docs, avgdl={bm25.avgdl:.2f}")

print('-'*120)

# Single query demo
q = next(dataset.queries_iter())
top10 = bm25.retrieve(q.text, k=10)
print("Query ID:", q.query_id, "Query Text:", q.text)
print('-'*50)
print("rank", "doc_id",  "score",  "title")
print('-'*50)
for rank, (doc_id, score) in enumerate(top10, 1):
    title = bm25.docs_store[doc_id]["title"]
    print(f"{rank:02d} {doc_id}  {score:.4f}  {title[:80]}")

## Task 3: BM25 Evaluator

### Task 3a: Implement evaluate function for "recall", "precision", "mrr", "ndcg"

In [ ]:
def evaluate(
    self,
    queries_iter: Iterable,
    qrels_iter: Iterable,
    k: int = 10,
    metrics: Optional[List[str]] = None,
) -> Dict[str, float]:
    """
    Compute evaluation metrics averaged over all queries.

    Args:
        queries_iter : iterable of queries (each has .query_id and .text)
        qrels_iter   : iterable of relevance judgments (each has .query_id, .doc_id, .relevance)
        k            : number of top documents to evaluate
        metrics      : list of metrics to compute, e.g. ["recall", "precision", "mrr", "ndcg"]
                       if None, compute all four

    Returns:
        Dict mapping metric name -> average score across all queries

    Note:
        Relevance:
            - treat relevance <= 0 as non-relevant (0)
            - treat relevance  > 0 as relevant (1)

        Metrics:
            Recall@k    = (number of relevant docs in top-k) / (total relevant docs for query)

            Precision@k = (number of relevant docs in top-k) / k

            MRR@k       = 1 / (rank of first relevant doc in top-k)
                        0 if no relevant doc found

            nDCG@k      = DCG@k / IDCG@k
                        DCG@k  = sum over rank i: rel_i / log2(i + 2)    (i is 0-indexed)
                        IDCG@k = DCG of the ideal ranking (sort all relevant docs by relevance desc)

    Hints:
        - define two helper functions inside evaluate():
            * normalize_relevance(rel) to convert relevance scores to 0 or 1 (binary)
            * dcg(rels) to compute DCG for a ranked relevance list
        - build a qrels dict: qrels[query_id][doc_id] = normalized relevance
        - skip queries that have no relevance judgments
        - use self.retrieve(query_text, k) to get ranked (doc_id, score) pairs
        - for duplicate qrel entries keep the maximum relevance
    """
        
    if metrics is None:
        metrics = ["recall", "precision", "mrr", "ndcg"]

    #TODO: your code here
    pass

BM25.evaluate = evaluate

### Task 3b: Grid Search

Implement and perform grid search for k1 ∈ {0.60, 1.20, 1.80} and b ∈ {0.25, 0.50, 0.75}

In [ ]:
def grid_search(
    self,
    queries_iter_factory,
    qrels_iter_factory,
    k: int = 10,
    k1_values: Tuple = (0.60, 1.20, 1.80),
    b_values: Tuple = (0.25, 0.50, 0.75),
) -> List[Dict]:
    """
    Evaluate all combinations of k1 and b, return results as a list of dicts.

    Args:
        queries_iter_factory : callable that returns a fresh queries iterator each time
        qrels_iter_factory   : callable that returns a fresh qrels iterator each time
        k                    : number of top documents to evaluate
        k1_values            : values of k1 to try
        b_values             : values of b to try

    Returns:
        List of dicts, one per (k1, b) combination:
        [
            {"k1": 0.60, "b": 0.25, "recall": ..., "precision": ..., "mrr": ..., "ndcg": ...},
            ...
        ]

    Note:
        - update self.k1 and self.b directly before each self.evaluate() call
        - pass factories/lambdas not iterators as ir_datasets iterators are single-pass,
          so call queries_iter_factory() and qrels_iter_factory() inside the loop to get a fresh iterator each time
    """

    # TODO: your code here

    pass

# attach to BM25 class
BM25.grid_search = grid_search

#### Run the cell below to perform the grid search

In [ ]:
# Grid search
# Make sure you have run the Check Task 2 cell first to initialize bm25
print("\nRunning grid search...")
results = bm25.grid_search(
    queries_iter_factory=dataset.queries_iter,
    qrels_iter_factory=dataset.qrels_iter,
    k=10,
)

# Display results table
import pandas as pd

df = pd.DataFrame(results)
df = df.set_index(["k1", "b"])
df.columns = ["Recall@10", "Precision@10", "MRR@10", "nDCG@10"] # change if your metric order is different
print("\n", df.to_string(float_format="%.4f"))




#### TODO: Report and justify your choice of best configuration



**Which do you consider your best configuration:** ...

**Why:** ...

## Task 4: Error Analysis

### TODO:
### Pick two queries, one where BM25 retrieved great documents, one where BM25 failed the retrieval.


#### Query 1 — Successful Retrieval

**Query ID:** ... 

**Query Text:** ...

**Top 5 Retrieved Documents:** ...


**Does the ranking make sense?** ...


**Proposed Improvement** ...

#### Query 2 — Unsuccessful Retrieval

**Query ID:** ... 

**Query Text:** ...

**Top 5 Retrieved Documents:** ...


**Does the ranking make sense?** ...


**Proposed Improvement** ...